In [0]:
%run ../../src/utils/config

In [0]:
%python
import os

config_path = os.path.abspath(
    os.path.join(os.getcwd(), "../../config/config.yml")
)

config = load_config(config_path)

In [0]:
catalog = config["target"]["catalog"]
silver_schema = config["target"]["silver_schema"]
gold_schema = config["target"]["gold_schema"]

print(f"Catalog: {catalog}")
print(f"Silver schema: {silver_schema}")
print(f"Gold schema: {gold_schema}")

required_tables = [
    f"{catalog}.{silver_schema}.pokemon_types",
    f"{catalog}.{silver_schema}.pokemon_abilities",
    f"{catalog}.{silver_schema}.pokemon_forms",
    f"{catalog}.{gold_schema}.dim_pokemon",
    f"{catalog}.{gold_schema}.dim_type",
    f"{catalog}.{gold_schema}.dim_ability",
    f"{catalog}.{gold_schema}.dim_form",
    f"{catalog}.{gold_schema}.bridge_pokemon_type",
    f"{catalog}.{gold_schema}.bridge_pokemon_ability",
    f"{catalog}.{gold_schema}.bridge_pokemon_form",
]

for table_name in required_tables:
    if not spark.catalog.tableExists(table_name):
        raise ValueError(f"Required table does not exist: {table_name}")

print("All required Silver and Gold tables exist.")

In [0]:
%python
from pyspark.sql import functions as F

silver_types = spark.table(
    f"{catalog}.{silver_schema}.pokemon_types"
)

dim_pokemon = (
    spark.table(f"{catalog}.{gold_schema}.dim_pokemon")
    .filter(F.col("is_current") == True)
    .select("pokemon_id", "pokemon_key")
)

dim_type = spark.table(
    f"{catalog}.{gold_schema}.dim_type"
).select("type_url", "type_key")

bridge_source = (
    silver_types
    .join(dim_pokemon, on="pokemon_id", how="left")
    .join(dim_type, on="type_url", how="left")
    .select(
        "pokemon_key",
        "type_key",
        F.col("slot").alias("type_slot")
    )
)

print(f"Silver relationship rows: {silver_types.count()}")
print(f"Prepared bridge rows: {bridge_source.count()}")

In [0]:
%python
missing_keys = bridge_source.filter(
    F.col("pokemon_key").isNull() |
    F.col("type_key").isNull()
).count()

if missing_keys > 0:
    raise ValueError(
        f"bridge_pokemon_type: {missing_keys} rows have unresolved "
        "pokemon_key or type_key."
    )

duplicate_rows = (
    bridge_source
    .groupBy("pokemon_key", "type_key", "type_slot")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if duplicate_rows > 0:
    raise ValueError(
        "bridge_pokemon_type: duplicate relationship keys found."
    )

print("Bridge source validation passed.")

In [0]:
%python
target_table = f"{catalog}.{gold_schema}.bridge_pokemon_type"

bridge_source.createOrReplaceTempView("src_bridge_pokemon_type")

spark.sql(f"""
    MERGE INTO {target_table} AS tgt
    USING src_bridge_pokemon_type AS src
       ON tgt.pokemon_key = src.pokemon_key
      AND tgt.type_key = src.type_key
      AND tgt.type_slot = src.type_slot

    WHEN NOT MATCHED THEN INSERT (
        pokemon_key,
        type_key,
        type_slot,
        gold_loaded_at
    )
    VALUES (
        src.pokemon_key,
        src.type_key,
        src.type_slot,
        current_timestamp()
    )
""")

print("bridge_pokemon_type load completed.")

In [0]:
%python
loaded_bridge = spark.table(
    f"{catalog}.{gold_schema}.bridge_pokemon_type"
)

null_key_count = loaded_bridge.filter(
    F.col("pokemon_key").isNull() |
    F.col("type_key").isNull()
).count()

if null_key_count > 0:
    raise ValueError(
        f"bridge_pokemon_type contains {null_key_count} null foreign keys."
    )

duplicate_count = (
    loaded_bridge
    .groupBy("pokemon_key", "type_key", "type_slot")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if duplicate_count > 0:
    raise ValueError("bridge_pokemon_type contains duplicate relationships.")

print(f"Loaded bridge rows: {loaded_bridge.count()}")
print("bridge_pokemon_type validation PASSED")

In [0]:
%python
from pyspark.sql import functions as F

silver_abilities = spark.table(
    f"{catalog}.{silver_schema}.pokemon_abilities"
)

dim_pokemon = (
    spark.table(f"{catalog}.{gold_schema}.dim_pokemon")
    .filter(F.col("is_current") == True)
    .select("pokemon_id", "pokemon_key")
)

dim_ability = spark.table(
    f"{catalog}.{gold_schema}.dim_ability"
).select("ability_url", "ability_key")

ability_source = (
    silver_abilities
    .join(dim_pokemon, on="pokemon_id", how="left")
    .join(dim_ability, on="ability_url", how="left")
    .select(
        "pokemon_key",
        "ability_key",
        F.col("slot").alias("ability_slot"),
        "is_hidden"
    )
)

print(f"Silver relationship rows: {silver_abilities.count()}")
print(f"Prepared bridge rows: {ability_source.count()}")

In [0]:
%python
missing_keys = ability_source.filter(
    F.col("pokemon_key").isNull() |
    F.col("ability_key").isNull()
).count()

if missing_keys > 0:
    raise ValueError(
        f"bridge_pokemon_ability: {missing_keys} rows have unresolved keys."
    )

duplicate_count = (
    ability_source
    .groupBy("pokemon_key", "ability_key", "ability_slot")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if duplicate_count > 0:
    raise ValueError(
        "bridge_pokemon_ability: duplicate relationship keys found."
    )

print("bridge_pokemon_ability source validation PASSED")

In [0]:
%python
target_table = f"{catalog}.{gold_schema}.bridge_pokemon_ability"

ability_source.createOrReplaceTempView("src_bridge_pokemon_ability")

spark.sql(f"""
    MERGE INTO {target_table} AS tgt
    USING src_bridge_pokemon_ability AS src
       ON tgt.pokemon_key = src.pokemon_key
      AND tgt.ability_key = src.ability_key
      AND tgt.ability_slot = src.ability_slot

    WHEN MATCHED AND NOT (tgt.is_hidden <=> src.is_hidden)
    THEN UPDATE SET
        tgt.is_hidden = src.is_hidden,
        tgt.gold_loaded_at = current_timestamp()

    WHEN NOT MATCHED THEN INSERT (
        pokemon_key,
        ability_key,
        ability_slot,
        is_hidden,
        gold_loaded_at
    )
    VALUES (
        src.pokemon_key,
        src.ability_key,
        src.ability_slot,
        src.is_hidden,
        current_timestamp()
    )
""")

print("bridge_pokemon_ability load completed.")

In [0]:
%python
loaded = spark.table(
    f"{catalog}.{gold_schema}.bridge_pokemon_ability"
)

missing_keys = loaded.filter(
    F.col("pokemon_key").isNull() |
    F.col("ability_key").isNull()
).count()

duplicate_count = (
    loaded
    .groupBy("pokemon_key", "ability_key", "ability_slot")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if missing_keys > 0:
    raise ValueError("bridge_pokemon_ability contains null foreign keys.")

if duplicate_count > 0:
    raise ValueError("bridge_pokemon_ability contains duplicate relationships.")

print(f"Loaded rows: {loaded.count()}")
print("bridge_pokemon_ability validation PASSED")

In [0]:
%python
from pyspark.sql import functions as F

silver_forms = spark.table(
    f"{catalog}.{silver_schema}.pokemon_forms"
)

dim_pokemon = (
    spark.table(f"{catalog}.{gold_schema}.dim_pokemon")
    .filter(F.col("is_current") == True)
    .select("pokemon_id", "pokemon_key")
)

dim_form = spark.table(
    f"{catalog}.{gold_schema}.dim_form"
).select("form_url", "form_key")

form_source = (
    silver_forms
    .join(dim_pokemon, on="pokemon_id", how="left")
    .join(dim_form, on="form_url", how="left")
    .select("pokemon_key", "form_key")
)

print(f"Silver relationship rows: {silver_forms.count()}")
print(f"Prepared bridge rows: {form_source.count()}")

In [0]:
%python
missing_keys = form_source.filter(
    F.col("pokemon_key").isNull() |
    F.col("form_key").isNull()
).count()

if missing_keys > 0:
    raise ValueError(
        f"bridge_pokemon_form: {missing_keys} rows have unresolved keys."
    )

duplicate_count = (
    form_source
    .groupBy("pokemon_key", "form_key")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if duplicate_count > 0:
    raise ValueError(
        "bridge_pokemon_form: duplicate relationship keys found."
    )

print("bridge_pokemon_form source validation PASSED")

In [0]:
%python
target_table = f"{catalog}.{gold_schema}.bridge_pokemon_form"

form_source.createOrReplaceTempView("src_bridge_pokemon_form")

spark.sql(f"""
    MERGE INTO {target_table} AS tgt
    USING src_bridge_pokemon_form AS src
       ON tgt.pokemon_key = src.pokemon_key
      AND tgt.form_key = src.form_key

    WHEN NOT MATCHED THEN INSERT (
        pokemon_key,
        form_key,
        gold_loaded_at
    )
    VALUES (
        src.pokemon_key,
        src.form_key,
        current_timestamp()
    )
""")

print("bridge_pokemon_form load completed.")

In [0]:
%python
loaded = spark.table(
    f"{catalog}.{gold_schema}.bridge_pokemon_form"
)

missing_keys = loaded.filter(
    F.col("pokemon_key").isNull() |
    F.col("form_key").isNull()
).count()

duplicate_count = (
    loaded
    .groupBy("pokemon_key", "form_key")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if missing_keys > 0:
    raise ValueError("bridge_pokemon_form contains null foreign keys.")

if duplicate_count > 0:
    raise ValueError("bridge_pokemon_form contains duplicate relationships.")

print(f"Loaded rows: {loaded.count()}")
print("bridge_pokemon_form validation PASSED")

In [0]:
%python
from pyspark.sql import functions as F

bridge_checks = {
    "bridge_pokemon_type": {
        "keys": ["pokemon_key", "type_key", "type_slot"],
        "foreign_keys": ["pokemon_key", "type_key"]
    },
    "bridge_pokemon_ability": {
        "keys": ["pokemon_key", "ability_key", "ability_slot"],
        "foreign_keys": ["pokemon_key", "ability_key"]
    },
    "bridge_pokemon_form": {
        "keys": ["pokemon_key", "form_key"],
        "foreign_keys": ["pokemon_key", "form_key"]
    }
}

for table_name, rules in bridge_checks.items():
    table = f"{catalog}.{gold_schema}.{table_name}"
    df = spark.table(table)

    null_fk_condition = F.lit(False)
    for key in rules["foreign_keys"]:
        null_fk_condition = null_fk_condition | F.col(key).isNull()

    null_fk_count = df.filter(null_fk_condition).count()

    duplicate_count = (
        df.groupBy(*rules["keys"])
        .count()
        .filter(F.col("count") > 1)
        .limit(1)
        .count()
    )

    null_load_count = df.filter(
        F.col("gold_loaded_at").isNull()
    ).count()

    if null_fk_count:
        raise ValueError(f"{table_name}: {null_fk_count} rows have null foreign keys.")

    if duplicate_count:
        raise ValueError(f"{table_name}: duplicate relationship keys found.")

    if null_load_count:
        raise ValueError(f"{table_name}: null gold_loaded_at values found.")

    print(f"{table_name}: rows={df.count()}, validation=PASSED")

print("All bridge validations passed.")